# figure 1
- task details including: barplots of corr/uncorr envs
- box image
- corr structure
- raster of trial-by-trial behavior

## imports

In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import permutations
from utils.plotSettings import *
from utils.opconNosepokeFunctions import *
# load df, for str and unstr
from utils.dfLoading import read_df
unstrdf = read_df('R:/behavioralData/processed/sessdf-2026-09-25.csv', task_type = ['unstr', 'str', 'dms', 'dls', 'sham'])

# Apply to sessdf
from utils.dfLoading import add_block_groups
unstrdf = add_block_groups(unstrdf)
unstrdf = unstrdf[~(unstrdf.rewarded_by_wrong_port==True)]   



## bar plots and corr matrices of corr/uncorr envs

In [206]:
np.array(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15]))).shape

(48, 4)

In [207]:
from matplotlib.colors import ListedColormap

# assign colors based on conditions
colors = {"str":"xkcd:emerald green", "unstr":"xkcd:barney purple"}
cmap_name = "bone"
cmap_range = (0.2, 0.9)   # part of the colormap to use (0 = start, 1 = end); (0, 1) is the full cmap
cmap = ListedColormap(plt.get_cmap(cmap_name)(np.linspace(*cmap_range, 256)))

x = [1,2,3,4]
prob_comb = np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0)
str_prob_comb = np.array([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])
unstr_prob_comb = prob_comb[~(prob_comb[:, None, :] == str_prob_comb[None, :, :]).all(axis = 2).any(axis = 1)]
prob_comb = np.vstack((str_prob_comb, unstr_prob_comb))
w = np.r_[np.full(4, (8/9)/4), np.full(32, (1/9)/32)]
cov = np.cov(prob_comb, rowvar=False, aweights=w)
d = np.sqrt(np.diag(cov)); str_corr_matrix = cov / np.outer(d, d)
from itertools import product
G = lambda mu, x: int(np.float32(0.7*np.exp(-(x-mu)**2/(2*0.875**2)) + 0.1)*100)
fw = []
for mu in range(1, 5):                       # SessionMean: random(1,5) -> 1..4
    for seq in product(range(4), repeat=4):  # every swap sequence of permutePorts
        p = [1, 2, 3, 4]
        for i, n in enumerate(seq):
            p[n], p[i] = p[i], p[n]
        fw.append([G(mu, x) for x in p])
unstr_corr_matrix = np.corrcoef(np.array(fw).T)


def env_bars(ax, probs, color, xlabel = False):
    ax.bar(x, probs, width = 1, color = color)
    ax.set_ylim(0, 80)
    ax.set_yticks([0, 80], [0, 80])
    ax.set_xticks(x, x)
    ax.set_ylabel("Reward %")
    if xlabel:
        ax.set_xlabel("Ports")

def corr_heatmap(ax, mat):
    # colorbar in an inset at the right edge of the (square) heatmap, so it is exactly as tall as the matrix
    cax = ax.inset_axes([1.05, 0, 0.06, 1])
    sns.heatmap(
        mat,
        cmap = cmap,
        ax = ax,
        cbar_ax = cax,
        vmin = -1,
        vmax = 1,
        annot = False,
        square = True,
        xticklabels = np.arange(1, 5, dtype = int),
        yticklabels = np.arange(1, 5, dtype = int)
        )
    # rotation 270 reads top-to-bottom; labelpad keeps it clear of the tick labels
    cax.set_ylabel('Correlation coefficient', rotation = 270, labelpad = 15)
    ax.set_xlabel("Ports")
    ax.set_ylabel("Ports")
    ax.tick_params(axis = 'y', rotation = 0)

# figure 1: bar plots in a 4x2 grid, structured in the top two rows, unstructured in the bottom two
# 1, 18, 35
str_rows = [0, 1, 2, 3]
unstr_rows = [1, 12, 18, 31]
figBars, axsBars = plt.subplots(4, 2, figsize = (6, 7), sharex = True)
for i in range(4):
    r, c = divmod(i, 2)   # fills row by row: (0,0), (0,1), (1,0), (1,1)
    env_bars(axsBars[r, c], str_prob_comb[str_rows[i]], colors["str"])
    env_bars(axsBars[r + 2, c], unstr_prob_comb[unstr_rows[i]], colors["unstr"], xlabel = r == 1)
plt.tight_layout()

# figure 2: correlation matrices
figMat, (matStr, matUnstr) = plt.subplots(1, 2, figsize = (9, 4))
corr_heatmap(matStr, str_corr_matrix)
corr_heatmap(matUnstr, unstr_corr_matrix)
plt.tight_layout()

## raster 

### unstructured

In [ ]:
animal = 'Onix'
box = 1
path = f"L:/4portProb_ephys/Box{box}_ephys/{animal}/Behavior/"      # for ephys only

files = exclude_files(get_files(path))
set_cwd(path)
files = [file for file in files if (file >=f'{animal}-*.dat')] 
files.sort()
print(files)
sessdf = trializer_v4_synced(files, animal)
sessdf['datetime'] = pd.to_datetime(sessdf.eptime, unit='s')
SESSION_GAP_SEC = 40 * 60

# if things look wonky maybe the data is not sorted!
# (it should be though the dat files are sorted per animal)
# sessdf = sessdf.sort_values(["animal", "eptime"]).copy()
gap_sec = sessdf.groupby("animal")["eptime"].diff()
new_session = gap_sec.isna() | (gap_sec > SESSION_GAP_SEC)

# Session numbers per animal: 1, 2, 3, ...
sessdf["session"] = new_session.groupby(sessdf["animal"]).cumsum().astype(int)
sessdf.rename(columns={'trial_rewprob':'rewprob'}, inplace = True)
# if you want to add block numbers, uncomment this section
sessdf['sess_block'] = 0
thing = sessdf.groupby(['animal', 'task', 'session']).datetime.head(1).astype('datetime64[ns]').diff().dt.seconds>2400
sessdf.loc[thing[thing == True].index, 'sess_block'] = 1
sessdf['sess_block'] = sessdf.groupby(['animal', 'task']).sess_block.cumsum()

['Bidoof-2026-02-24-10-24-27.dat', 'Bidoof-2026-02-25-10-24-27.dat', 'Bidoof-2026-02-26-10-24-27.dat', 'Bidoof-2026-02-27-10-24-27.dat', 'Bidoof-2026-02-28-10-24-27.dat', 'Bidoof-2026-03-01-10-24-27.dat', 'Bidoof-2026-03-02-10-24-27.dat', 'Bidoof-2026-03-03-10-24-27.dat', 'Bidoof-2026-03-04-10-24-27.dat', 'Bidoof-2026-03-05-10-24-27.dat', 'Bidoof-2026-03-06-10-24-27.dat', 'Bidoof-2026-03-07-10-24-27.dat', 'Bidoof-2026-03-08-10-24-27.dat', 'Bidoof-2026-03-09-10-24-27.dat', 'Bidoof-2026-03-10-10-24-27.dat', 'Bidoof-2026-03-10-16-42-57.dat', 'Bidoof-2026-03-11-16-42-57.dat', 'Bidoof-2026-03-11-20-02-48.dat', 'Bidoof-2026-03-12-20-02-48.dat', 'Bidoof-2026-03-13-20-02-48.dat', 'Bidoof-2026-03-14-20-02-48.dat', 'Bidoof-2026-03-15-20-02-48.dat', 'Bidoof-2026-03-16-20-02-48.dat', 'Bidoof-2026-03-17-20-02-48.dat', 'Bidoof-2026-03-18-20-02-48.dat', 'Bidoof-2026-03-19-20-02-48.dat', 'Bidoof-2026-03-20-20-02-48.dat', 'Bidoof-2026-03-21-20-02-48.dat', 'Bidoof-2026-03-22-08-24-15.dat', 'Bidoof-2026-

In [193]:
# this plots blockwise raster of choice
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle
from matplotlib import transforms
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib qt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgb

def make_seq_cmap(base, name=None, light='#fbf7fb', dark_factor=0.45):
    # sequential cmap: near-white -> base colour -> darker version of base (like 'Purples')
    base_rgb = np.array(to_rgb(base))
    return LinearSegmentedColormap.from_list(
        name or f'seq_{base}', [to_rgb(light), base_rgb, base_rgb * dark_factor])

band_cmap = make_seq_cmap('xkcd:barney purple', name='BarneyPurples')

# band_cmap = 'Purples'       # any sequential matplotlib cmap, e.g. 'Oranges'
band_range = (0.1, 0.4) # part of the cmap to use: lightest (lowest prob) -> darkest (highest prob)
prob_range = None   # reward % mapped to the ends of band_range; set to None to use the min/max prob in the data
rect_alpha = 0.7          # opaque bands keep the shades distinct; lower range above instead of alpha to lighten
r_tick_color = 'xkcd:black'       # rewarded: tall black tick
nr_tick_color = 'xkcd:black'   # unrewarded: short grey tick
rew_lw, unrew_lw = 1.1, 0.8 # lw
rew_h, unrew_h = 0.2, 0.1   # half-heights of the ticks
scalebar_frac = 0.1   # scale bar is the largest 1/2/5 x 10^k trials that fits in this fraction of the visible x-range
legend_loc = 'right'   # 'top': one row above the plot; 'right': one vertical column right of the plot

# one shade per reward probability, placed along the cmap in proportion to its value
rewprob_vals = sorted(sessdf['rewprob'].dropna().unique())
pmin, pmax = prob_range if prob_range is not None else (min(rewprob_vals), max(rewprob_vals))
frac = (np.array(rewprob_vals) - pmin) / ((pmax - pmin) or 1)
shades = plt.get_cmap(band_cmap)(band_range[0] + frac * (band_range[1] - band_range[0]))
rew_palette = {p: c for p, c in zip(rewprob_vals, shades)}

def nice_length(span, frac=scalebar_frac):
    # largest 1, 2 or 5 x 10^k that is <= frac * span
    target = max(span * frac, 1)
    mag = 10 ** np.floor(np.log10(target))
    for m in (5, 2, 1):
        if m * mag <= target:
            return m * mag
    return mag

class _VTick(Line2D):
    # vertical legend tick whose length is recomputed at draw time to match the raster ticks in ax
    def __init__(self, ax, half_h, cx, cy, **kw):
        super().__init__([cx, cx], [cy, cy], **kw)
        self._ax, self._half_h, self._cy = ax, half_h, cy

    def draw(self, renderer):
        y0, y1 = self._ax.get_ylim()
        px_per_unit = self._ax.bbox.height / abs(y1 - y0)
        half = self._half_h * px_per_unit / renderer.points_to_pixels(1.0)   # legend handles are in points
        self.set_ydata([self._cy - half, self._cy + half])
        super().draw(renderer)

class VTickHandler(HandlerBase):
    def __init__(self, ax, half_h):
        super().__init__()
        self.ax, self.half_h = ax, half_h

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        tick = _VTick(self.ax, self.half_h, width / 2 - xdescent, height / 2 - ydescent,
                      color=orig_handle.get_color(), lw=orig_handle.get_linewidth(), transform=trans)
        return [tick]

def add_raster_legend(ax, loc=None):
    # reward-prob bands (match the background) + vertical outcome ticks (same size as in the raster)
    loc = loc or legend_loc
    probs = list(rew_palette.items())
    if loc == 'right':
        probs = probs[::-1]   # highest prob on top
    handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in probs]
    rew_handle = Line2D([], [], color=r_tick_color, lw=rew_lw, label='Rewarded')
    unrew_handle = Line2D([], [], color=nr_tick_color, lw=unrew_lw, label='Unrewarded')
    handler_map = {rew_handle: VTickHandler(ax, rew_h), unrew_handle: VTickHandler(ax, unrew_h)}
    # make each legend row tall enough for the rewarded tick (estimated from the current axes size)
    fontsize = plt.rcParams['legend.fontsize']
    fontsize = fontsize if isinstance(fontsize, (int, float)) else plt.rcParams['font.size']
    y0, y1 = ax.get_ylim()
    tick_pts = 2 * rew_h * ax.bbox.height / abs(y1 - y0) * 72 / ax.figure.dpi
    handleheight = max(0.7, tick_pts / fontsize)
    common = dict(handles=handles + [rew_handle, unrew_handle], handler_map=handler_map,
                  frameon=False, handlelength=1.2, handleheight=handleheight)
    if loc == 'right':
        return ax.legend(ncol=1, loc='center left', bbox_to_anchor=(1.01, 0.5), labelspacing=0.4, **common)
    return ax.legend(ncol=len(handles) + 2, loc='lower center', bbox_to_anchor=(0.5, 1.02),
                     columnspacing=1, **common)

# Create a function to update the plot based on user input
def update_plot(session_range):
    sessdf_filtered = sessdf[sessdf['block_num'].between(*session_range)].copy()
    fig, ax = plt.subplots(figsize=(14, 3))
    # tick height encodes outcome, all centred on the port
    rew = sessdf_filtered[sessdf_filtered['reward'] == 1]
    unrew = sessdf_filtered[sessdf_filtered['reward'] == 0]
    ax.vlines(rew['trial'], rew['port']-rew_h, rew['port']+rew_h, color=r_tick_color, lw=rew_lw, zorder=3)
    ax.vlines(unrew['trial'], unrew['port']-unrew_h, unrew['port']+unrew_h, color=nr_tick_color, lw=unrew_lw, zorder=3)

    last = sessdf.groupby('block_num')['trial'].max().to_list()
    start = sessdf.groupby('block_num')['trial'].min().to_list()
    # per-block list of reward probs for ports 1-4
    block_probs = [[float(j) for j in i.strip('[]').split()] for i in sessdf.groupby('block_num').head(1).rewprobfull]
    for ind, l in enumerate(last):
        # background bands, slightly shorter than 1 so each port row has a white gap
        for port, p in enumerate(block_probs[ind], start=1):
            if p in rew_palette:
                ax.add_patch(Rectangle((start[ind]-0.5, port-0.4), l-start[ind]+1, 0.8,
                                       fc=rew_palette[p], alpha=rect_alpha, ec='none', zorder=0))
        ax.axvline(l+0.5, color='white', lw=1, zorder=1)

    ax.set_ylim(0.5, 4.5)
    ax.invert_yaxis()   # port 1 on top
    ax.set_yticks([1, 2, 3, 4], ['Port 1', 'Port 2', 'Port 3', 'Port 4'])
    ax.tick_params(left=False)
    ax.set_xticks([])
    sns.despine(ax=ax, left=True, bottom=True)

    # scale bar instead of an x-axis: x in data units, y in axes units, so it stays
    # below the plot and is redrawn to a sensible length whenever you zoom or pan
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    bar, = ax.plot([], [], color='k', lw=1.5, transform=trans, clip_on=False)
    bar_label = ax.text(0, -0.1, '', transform=trans, ha='center', va='top', fontsize='small')

    def update_scalebar(ax):
        x0, x1 = ax.get_xlim()
        n = nice_length(x1 - x0)
        xs = x0 + 0.01 * (x1 - x0)
        bar.set_data([xs, xs + n], [-0.05, -0.05])
        bar_label.set_position((xs + n / 2, -0.1))
        bar_label.set_text(f'{n:g} trial' + ('' if n == 1 else 's'))

    ax.callbacks.connect('xlim_changed', update_scalebar)
    x_limit = (sessdf_filtered['trial'].min() - 0.5, sessdf_filtered['trial'].max() + 0.5)
    ax.set_xlim(*x_limit)   # triggers update_scalebar

    add_raster_legend(ax)
    plt.tight_layout()
    plt.show()

# Define the session range using a widget
session_range_slider = widgets.IntRangeSlider(
    value=[251, 253],
    min=sessdf['block_num'].min(),
    max=sessdf['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)

# Create an interactive plot using interactive
interactive_plot = widgets.interactive(update_plot, session_range=session_range_slider)

# Display the interactive plot
display(interactive_plot)

interactive(children=(IntRangeSlider(value=(251, 253), continuous_update=False, description='Block Range:', ma…

In [ ]:
# same choice raster as above, plus a lower axis with each block's reward probs as bars (ports 1-4 left to right)
# run the cell above first: reuses its settings, rew_palette and nice_length
bar_width = 10   # width of each port's bar, in trials (same for every block)
bar_gap = 2     # gap between neighbouring port bars, in trials; the group of 4 is centred on its block

def update_plot_bars(session_range):
    sessdf_filtered = sessdf[sessdf['block_num'].between(*session_range)].copy()
    fig, (ax, bax) = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True, height_ratios=[3, 1.2])
    # tick height encodes outcome, all centred on the port
    rew = sessdf_filtered[sessdf_filtered['reward'] == 1]
    unrew = sessdf_filtered[sessdf_filtered['reward'] == 0]
    ax.vlines(rew['trial'], rew['port']-rew_h, rew['port']+rew_h, color=r_tick_color, lw=rew_lw, zorder=3)
    ax.vlines(unrew['trial'], unrew['port']-unrew_h, unrew['port']+unrew_h, color=nr_tick_color, lw=unrew_lw, zorder=3)

    last = sessdf.groupby('block_num')['trial'].max().to_list()
    start = sessdf.groupby('block_num')['trial'].min().to_list()
    # per-block list of reward probs for ports 1-4
    block_probs = [[float(j) for j in i.strip('[]').split()] for i in sessdf.groupby('block_num').head(1).rewprobfull]
    bar_x, bar_h, bar_c = [], [], []
    for ind, l in enumerate(last):
        # background bands, slightly shorter than 1 so each port row has a white gap
        for port, p in enumerate(block_probs[ind], start=1):
            if p in rew_palette:
                ax.add_patch(Rectangle((start[ind]-0.5, port-0.4), l-start[ind]+1, 0.8,
                                       fc=rew_palette[p], alpha=rect_alpha, ec='none', zorder=0))
        # lower axis: fixed-width bars, group centred on the block, bar height = reward %
        centre = (start[ind] + l) / 2
        n_ports = len(block_probs[ind])
        for port, p in enumerate(block_probs[ind], start=1):
            bar_x.append(centre + (port - (n_ports + 1) / 2) * (bar_width + bar_gap))
            bar_h.append(p)
            bar_c.append(rew_palette.get(p, 'lightgrey'))
        ax.axvline(l+0.5, color='white', lw=1, zorder=1)
        bax.axvline(l+0.5, color='lightgrey', lw=0.5, zorder=0)
    # same alpha as the raster bands so the shades match exactly
    bax.bar(bar_x, bar_h, width=bar_width, color=bar_c, alpha=rect_alpha, ec='none', zorder=2)

    ax.set_ylim(0.5, 4.5)
    ax.invert_yaxis()   # port 1 on top
    ax.set_yticks([1, 2, 3, 4], ['Port 1', 'Port 2', 'Port 3', 'Port 4'])
    ax.tick_params(left=False, bottom=False)
    sns.despine(ax=ax, left=True, bottom=True)

    bax.set_ylim(0, 80)
    bax.set_yticks([0, 40, 80])
    bax.set_ylabel('Rew. %\n(ports 1-4)', fontsize='small')
    bax.set_xticks([])
    sns.despine(ax=bax, bottom=True)

    # scale bar under the raster, redrawn on zoom/pan
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    bar, = ax.plot([], [], color='k', lw=1.5, transform=trans, clip_on=False)
    bar_label = ax.text(0, -0.08, '', transform=trans, ha='center', va='top', fontsize='small')

    def update_scalebar(a):
        x0, x1 = a.get_xlim()
        n = nice_length(x1 - x0)
        xs = x0 + 0.01 * (x1 - x0)
        bar.set_data([xs, xs + n], [-0.04, -0.04])
        bar_label.set_position((xs + n / 2, -0.08))
        bar_label.set_text(f'{n:g} trial' + ('' if n == 1 else 's'))

    ax.callbacks.connect('xlim_changed', update_scalebar)
    x_limit = (sessdf_filtered['trial'].min() - 0.5, sessdf_filtered['trial'].max() + 0.5)
    ax.set_xlim(*x_limit)   # shared x, triggers update_scalebar

    # handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in rew_palette.items()]
    # ax.legend(handles=handles, ncol=len(handles), loc='lower center',
    #           bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in rew_palette.items()]
    outcome_handles = [Line2D([], [], color=r_tick_color, lw=rew_lw*1., label='Rewarded'),
                        Line2D([], [], color=nr_tick_color, lw=unrew_lw*1., label='Unrewarded')]
    ax.legend(handles=handles + outcome_handles, ncol=len(handles) + 2, loc='lower center',
                bbox_to_anchor=(0.5, 1.02), frameon=False, handlelength=1.2, columnspacing=1)
    plt.tight_layout()
    fig.subplots_adjust(hspace=0.35)   # room for the scale bar between the two axes
    plt.show()

session_range_slider_bars = widgets.IntRangeSlider(
    value=[251, 253],
    min=sessdf['block_num'].min(),
    max=sessdf['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)
display(widgets.interactive(update_plot_bars, session_range=session_range_slider_bars))

### structured

In [165]:
hsb_6 = pd.read_csv(r"L:\4portProb_structured\HSB_F_006.csv")
hsb_7 = pd.read_csv(r"L:\4portProb_structured\HSB_F_007.csv")
hsb_8 = pd.read_csv(r"L:\4portProb_structured\HSB_F_008.csv")
hsb_9 = pd.read_csv(r"L:\4portProb_structured\HSB_F_009.csv")
hsb_10 = pd.read_csv(r"L:\4portProb_structured\HSB_F_010.csv")
hsb_11 = pd.read_csv(r"L:\4portProb_structured\HSB_F_011.csv")
hsb_12 = pd.read_csv(r"L:\4portProb_structured\HSB_F_012.csv")
hsb_13 = pd.read_csv(r"L:\4portProb_structured\HSB_F_013.csv")
hsb_14 = pd.read_csv(r"L:\4portProb_structured\HSB_F_014.csv")

hsb_6.animal = 'HSB_F_006'
hsb_7.animal = 'HSB_F_007'
hsb_8.animal = 'HSB_F_008'
hsb_9.animal = 'HSB_F_009'
hsb_10.animal = 'HSB_F_010'
hsb_11.animal = 'HSB_F_011'
hsb_12.animal = 'HSB_F_012'
hsb_13.animal = 'HSB_F_013'
hsb_14.animal = 'HSB_F_014'

hsb_data = pd.concat([hsb_6, hsb_7, hsb_8, hsb_9, hsb_10, hsb_11, hsb_12, hsb_13, hsb_14]).reset_index()
hsb_data['trial'] = np.arange(hsb_data.shape[0])
hsb_data['env'] = hsb_data['task']
hsb_data['task'] = 'str'
hsb_data['datetime'] = pd.to_datetime(hsb_data.eptime, unit = 's')
from utils.dfLoading import add_block_groups
hsb_data = hsb_data.rename(columns = {'session':'block_num'})#.drop(columns = "Unnamed: 0")
hsb_data = add_block_groups(hsb_data)
hsb_data = hsb_data.rename(columns = {'sess_block': 'session'})#.drop(columns = "Unnamed: 0")
# ## week relative to each animal-task first date
hsb_data['week_task'] = hsb_data.groupby(['animal', 'task'])['datetime'].transform(
    lambda x: ((x - x.min()).dt.days // 7) + 1
)
hsb_data['day_task'] = hsb_data.groupby('animal')['datetime'].transform(
    lambda x: ((x - x.min()).dt.days) + 1
)
from utils.trializer_v4 import apply_task_rules_from_json
rules_path = r"C:\Users\dlab\banditAnalysis\utils\task_data.json"
hsb_data = apply_task_rules_from_json(hsb_data, rules_path=rules_path)
struc_df = hsb_data[(hsb_data.animal == 'HSB_F_008') & (hsb_data.task == 'str')]

In [184]:
# this plots blockwise raster of choice
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle
from matplotlib import transforms
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib qt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
import numpy as np

band_cmap = 'Greens'       # any sequential matplotlib cmap, e.g. 'Oranges'
band_range = (0.2, 0.8) # part of the cmap to use: lightest (lowest prob) -> darkest (highest prob)
prob_range = None   # reward % mapped to the ends of band_range; set to None to use the min/max prob in the data
rect_alpha = 0.7          # opaque bands keep the shades distinct; lower range above instead of alpha to lighten
r_tick_color = 'xkcd:black'       # rewarded: tall black tick
nr_tick_color = 'xkcd:black'   # unrewarded: short grey tick
rew_lw, unrew_lw = 1.1, 0.8 # lw
rew_h, unrew_h = 0.2, 0.1   # half-heights of the ticks
scalebar_frac = 0.1   # scale bar is the largest 1/2/5 x 10^k trials that fits in this fraction of the visible x-range
legend_loc = 'right'   # 'top': one row above the plot; 'right': one vertical column right of the plot

# one shade per reward probability, placed along the cmap in proportion to its value
rewprob_vals = sorted(struc_df['rewprob'].dropna().unique())
pmin, pmax = prob_range if prob_range is not None else (min(rewprob_vals), max(rewprob_vals))
frac = (np.array(rewprob_vals) - pmin) / ((pmax - pmin) or 1)
shades = plt.get_cmap(band_cmap)(band_range[0] + frac * (band_range[1] - band_range[0]))
rew_palette = {p: c for p, c in zip(rewprob_vals, shades)}

def nice_length(span, frac=scalebar_frac):
    # largest 1, 2 or 5 x 10^k that is <= frac * span
    target = max(span * frac, 1)
    mag = 10 ** np.floor(np.log10(target))
    for m in (5, 2, 1):
        if m * mag <= target:
            return m * mag
    return mag

class _VTick(Line2D):
    # vertical legend tick whose length is recomputed at draw time to match the raster ticks in ax
    def __init__(self, ax, half_h, cx, cy, **kw):
        super().__init__([cx, cx], [cy, cy], **kw)
        self._ax, self._half_h, self._cy = ax, half_h, cy

    def draw(self, renderer):
        y0, y1 = self._ax.get_ylim()
        px_per_unit = self._ax.bbox.height / abs(y1 - y0)
        half = self._half_h * px_per_unit / renderer.points_to_pixels(1.0)   # legend handles are in points
        self.set_ydata([self._cy - half, self._cy + half])
        super().draw(renderer)

class VTickHandler(HandlerBase):
    def __init__(self, ax, half_h):
        super().__init__()
        self.ax, self.half_h = ax, half_h

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        tick = _VTick(self.ax, self.half_h, width / 2 - xdescent, height / 2 - ydescent,
                      color=orig_handle.get_color(), lw=orig_handle.get_linewidth(), transform=trans)
        return [tick]

def add_raster_legend(ax, loc=None):
    # reward-prob bands (match the background) + vertical outcome ticks (same size as in the raster)
    loc = loc or legend_loc
    probs = list(rew_palette.items())
    if loc == 'right':
        probs = probs[::-1]   # highest prob on top
    handles = [Patch(fc=c, alpha=rect_alpha, ec='none', label=f'{p:g}%') for p, c in probs]
    rew_handle = Line2D([], [], color=r_tick_color, lw=rew_lw, label='Rewarded')
    unrew_handle = Line2D([], [], color=nr_tick_color, lw=unrew_lw, label='Unrewarded')
    handler_map = {rew_handle: VTickHandler(ax, rew_h), unrew_handle: VTickHandler(ax, unrew_h)}
    # make each legend row tall enough for the rewarded tick (estimated from the current axes size)
    fontsize = plt.rcParams['legend.fontsize']
    fontsize = fontsize if isinstance(fontsize, (int, float)) else plt.rcParams['font.size']
    y0, y1 = ax.get_ylim()
    tick_pts = 2 * rew_h * ax.bbox.height / abs(y1 - y0) * 72 / ax.figure.dpi
    handleheight = max(0.7, tick_pts / fontsize)
    common = dict(handles=handles + [rew_handle, unrew_handle], handler_map=handler_map,
                  frameon=False, handlelength=1.2, handleheight=handleheight)
    if loc == 'right':
        return ax.legend(ncol=1, loc='center left', bbox_to_anchor=(1.01, 0.5), labelspacing=0.4, **common)
    return ax.legend(ncol=len(handles) + 2, loc='lower center', bbox_to_anchor=(0.5, 1.02),
                     columnspacing=1, **common)

# Create a function to update the plot based on user input
def update_plot(session_range):
    struc_df_filtered = struc_df[struc_df['block_num'].between(*session_range)].copy()
    fig, ax = plt.subplots(figsize=(14, 3))
    # tick height encodes outcome, all centred on the port
    rew = struc_df_filtered[struc_df_filtered['reward'] == 1]
    unrew = struc_df_filtered[struc_df_filtered['reward'] == 0]
    ax.vlines(rew['trial'], rew['port']-rew_h, rew['port']+rew_h, color=r_tick_color, lw=rew_lw, zorder=3)
    ax.vlines(unrew['trial'], unrew['port']-unrew_h, unrew['port']+unrew_h, color=nr_tick_color, lw=unrew_lw, zorder=3)

    last = struc_df.groupby('block_num')['trial'].max().to_list()
    start = struc_df.groupby('block_num')['trial'].min().to_list()
    # per-block list of reward probs for ports 1-4
    block_probs = [[float(j) for j in i.strip('[]').split()] for i in struc_df.groupby('block_num').head(1).rewprobfull]
    for ind, l in enumerate(last):
        # background bands, slightly shorter than 1 so each port row has a white gap
        for port, p in enumerate(block_probs[ind], start=1):
            if p in rew_palette:
                ax.add_patch(Rectangle((start[ind]-0.5, port-0.4), l-start[ind]+1, 0.8,
                                       fc=rew_palette[p], alpha=rect_alpha, ec='none', zorder=0))
        ax.axvline(l+0.5, color='white', lw=1, zorder=1)

    ax.set_ylim(0.5, 4.5)
    ax.invert_yaxis()   # port 1 on top
    ax.set_yticks([1, 2, 3, 4], ['Port 1', 'Port 2', 'Port 3', 'Port 4'])
    ax.tick_params(left=False)
    ax.set_xticks([])
    sns.despine(ax=ax, left=True, bottom=True)

    # scale bar instead of an x-axis: x in data units, y in axes units, so it stays
    # below the plot and is redrawn to a sensible length whenever you zoom or pan
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    bar, = ax.plot([], [], color='k', lw=1.5, transform=trans, clip_on=False)
    bar_label = ax.text(0, -0.1, '', transform=trans, ha='center', va='top', fontsize='small')

    def update_scalebar(ax):
        x0, x1 = ax.get_xlim()
        n = nice_length(x1 - x0)
        xs = x0 + 0.01 * (x1 - x0)
        bar.set_data([xs, xs + n], [-0.05, -0.05])
        bar_label.set_position((xs + n / 2, -0.1))
        bar_label.set_text(f'{n:g} trial' + ('' if n == 1 else 's'))

    ax.callbacks.connect('xlim_changed', update_scalebar)
    x_limit = (struc_df_filtered['trial'].min() - 0.5, struc_df_filtered['trial'].max() + 0.5)
    ax.set_xlim(*x_limit)   # triggers update_scalebar

    add_raster_legend(ax)
    plt.tight_layout()
    plt.show()

# Define the session range using a widget
session_range_slider = widgets.IntRangeSlider(
    value=[1070, 1073],
    min=struc_df['block_num'].min(),
    max=struc_df['block_num'].max(),
    step=1,
    description='Block Range:',
    continuous_update=False
)

# Create an interactive plot using interactive
interactive_plot = widgets.interactive(update_plot, session_range=session_range_slider)

# Display the interactive plot
display(interactive_plot)

interactive(children=(IntRangeSlider(value=(1070, 1073), continuous_update=False, description='Block Range:', …

In [185]:
struc_df.loc[774436, :]

index                       193711
trial                       774436
trialstart                281687.0
port                           4.0
reward                         1.0
trialend                  282226.0
block_num                     1070
rewprob                       80.0
eptime                1773255892.0
task                           str
rewprobfull          [10 15 46 80]
animal                   HSB_F_008
datetime       2026-03-11 19:04:52
session                        188
env                            str
block_group                      1
week_task                        9
day_task                        63
Name: 774436, dtype: object